# A.1 給幾個例子能改變回答嗎？


「請把3轉成另一個數字」不夠明確：翻倍、平方都說得通。先給1→2、2→4，再問3，能讓讀者猜到這次想用翻倍，紙上預期6；換成1→3、2→6，則預期9。例子和問題一起成為本次輸入，權重沒有因此更新。

這種在當前前文提供任務線索的方式叫上下文學習。沒有示例叫zero-shot，幾個示例叫few-shot；shot數提示中的例子，不數訓練步數。有限示例仍可能有多種規律，因此人的預期需要與模型的實際回答分開。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
question = "請依本次規則回答：3→?"
examples = ["1→2", "2→4"]
zero = question
few = "示例：" + "；".join(examples) + "。\n" + question
changed = "示例：1→3；2→6。\n" + question
print("無例子：", zero)
print("兩例子：", few)
print("換規則：", changed)

無例子： 請依本次規則回答：3→?
兩例子： 示例：1→2；2→4。
請依本次規則回答：3→?
換規則： 示例：1→3；2→6。
請依本次規則回答：3→?


`join`以分號接例子，換行後放同一問題，程式展開三份提示，沒有模型。它讓我們先核對改的是示例，不是偷偷換了一道題。

若用模型比較，固定權重、問題與生成設定，只改示例，另留新數值與組合。交換例子順序可能改變輸出，也要記錄。把示例從提示移除後，沒有保證仍會沿用這次規則；SFT會更新權重，是另一種工作。

<details>
<summary>補充：舊小模型的ASCII示例映射結果</summary>

課程實測另讓一個小模型真正學過「加上本次偏移量，再取個位數」的ASCII示例。ASCII是一套從0到127的基本字元編號，包含英文字母、數字與常見符號；UTF-8保存這些字元時，每個字元只用一個byte。下面正式提示用`->`兩個符號表示箭頭，與前面示例的一個`→`字元不同，不能把兩份提示的編碼混為一談。輸入數值按家族分成六個訓練、一個驗證、一個最後檢查家族；最後檢查只用數值3，不能當成廣泛的上下文學習評測。同一份已訓練權重、每次取最高分token，實際得到：

| 本次提供的示例 | 應答 | 真實生成 |
| --- | --- | --- |
| 沒有示例 | `UNKNOWN` | `UNKNOWN` |
| `0->2;1->3` | `5` | `9` |
| 改成 `0->5;1->6` | `8` | `9` |
| 交換成 `1->3;0->2` | `5` | `8` |

三份有示例的回答都錯，交換順序還改變輸出。這次確實改了輸入，也確實讓模型生成；結果沒有支持它已學會依新示例運用規則。四份完整提示、原始ID與EOS保存在[正式報告的ICL紀錄](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/rag.json)，不要把前面的人造翻倍示例當作這份成績。

</details>
